# 🛠️ Preparación del entorno · Unidad 2 (Clases 2, 3 y 4)

**Ejecuta este notebook en casa, ANTES de la clase, con buena conexión a internet.**

Hace todo por ti:
1. Revisa tu Python, tu memoria RAM y tu espacio en disco.
2. Instala las librerías que usaremos.
3. Descarga los **5 modelos de IA** de la unidad (~3,3 GB en total) y los guarda en tu computador.
4. Prueba que cada modelo carga y que MiniGPT (Qwen3) responde.
5. Te muestra una tabla final: si todo sale con ✅, estás listo.

⏱️ **Tiempo:** 15–40 minutos, según tu internet. Solo se hace **una vez**: los modelos quedan guardados y en clase cargan en segundos.

▶️ **Cómo usarlo:** menú *Run → Run All* (o *Ejecutar todo*) y espera a que termine la última celda.

## 1. Revisar tu computador

In [ ]:
import sys, platform, shutil, os
from pathlib import Path

print(f"Python      : {sys.version.split()[0]}  ({sys.executable})")
print(f"Sistema     : {platform.system()} {platform.release()}")
print(f"Núcleos CPU : {os.cpu_count()}")

if sys.version_info < (3, 10):
    print("\n⚠️ Tu Python es menor a 3.10. Las versiones nuevas de transformers necesitan 3.10 o superior.")
    print("   Crea un entorno nuevo, por ejemplo:  conda create -n ia python=3.11  y abre este notebook con ese kernel.")
else:
    print("\n✅ Versión de Python compatible.")

# Espacio libre donde se guardan los modelos (caché de Hugging Face)
cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
cache.mkdir(parents=True, exist_ok=True)
libre_gb = shutil.disk_usage(cache).free / 1e9
print(f"\nCarpeta de modelos: {cache}")
print(f"Espacio libre     : {libre_gb:.1f} GB")
print("✅ Espacio suficiente." if libre_gb >= 6 else "⚠️ Necesitas al menos ~6 GB libres (modelos + librerías). Libera espacio antes de seguir.")

## 2. Instalar librerías

In [ ]:
import importlib.util, subprocess

paquetes = {
    "torch": "torch",
    "transformers": "transformers>=4.51",   # 4.51+ es necesario para Qwen3
    "datasets": "datasets",
    "accelerate": "accelerate",
    "peft": "peft",                          # LoRA (Clase 4)
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "psutil": "psutil",
    "huggingface_hub": "huggingface_hub",
}
INSTALAR_GRADIO = False   # ← cámbialo a True si quieres la interfaz web opcional de la Clase 3

faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]
if INSTALAR_GRADIO and importlib.util.find_spec("gradio") is None:
    faltan.append("gradio")

if faltan:
    print("Instalando:", ", ".join(faltan), "\n(puede tardar varios minutos la primera vez)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    print("✅ Instalación terminada. Si la siguiente celda falla al importar, reinicia el kernel y ejecuta todo de nuevo.")
else:
    print("✅ Todas las librerías ya estaban instaladas.")

In [ ]:
import gc, time, warnings
import psutil, torch, transformers
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

ram_gb = psutil.virtual_memory().total / 1e9
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(f"RAM total: {ram_gb:.1f} GB")
if ram_gb < 7.5:
    print("⚠️ Tienes poca RAM. Funcionará, pero en clase cierra el navegador y otros programas.")
    print("   En los notebooks puedes usar DTYPE = torch.bfloat16 para gastar la mitad de memoria.")
else:
    print("✅ RAM suficiente.")

def version(v):
    return tuple(int(x) for x in v.split("+")[0].split(".")[:2])
if version(torch.__version__) < (2, 6):
    print("⚠️ torch es anterior a 2.6: algunos modelos antiguos (.bin) no cargarán. Actualiza con:  pip install -U torch")
if version(transformers.__version__) < (4, 51):
    print("⚠️ transformers es anterior a 4.51: Qwen3 no funcionará. Actualiza con:  pip install -U transformers")

if torch.cuda.is_available():
    print("🚀 GPU NVIDIA detectada:", torch.cuda.get_device_name(0))
else:
    print("💻 Sin GPU: todo correrá en CPU (está pensado para eso).")

## 3. Descargar los modelos

| # | Modelo | Clase | Para qué lo usamos | Tamaño |
|---|---|---|---|---|
| 1 | `datificate/gpt2-small-spanish` | 2 | GPT-2 en español: lo convertimos en **MiniGPT** con fine-tuning | ~500 MB |
| 2 | `dccuchile/bert-base-spanish-wwm-cased` | 2 y 4 | **BETO** (BERT en español): rellenar palabras y clasificador propio | ~440 MB |
| 3 | `Qwen/Qwen3-0.6B` | 3 y 4 | El **nuevo cerebro de MiniGPT**: chat, prompts, razonamiento, LoRA | ~1,5 GB |
| 4 | `mrm8488/bert-spanish-cased-finetuned-ner` | 4 | Detectar **personas, lugares y organizaciones** (NER) | ~420 MB |
| 5 | `pysentimiento/robertuito-sentiment-analysis` | 4 | **Sentimiento** de textos en español | ~435 MB |

Cada modelo se descarga **cargándolo igual que en clase**: así se bajan exactamente los archivos que vamos a usar. Si uno falla (por ejemplo, se cortó internet), simplemente vuelve a ejecutar esta celda: continúa donde quedó.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM,
                          AutoModelForTokenClassification, AutoModelForSequenceClassification)

MODELOS = [
    ("datificate/gpt2-small-spanish",               AutoModelForCausalLM,               "Clase 2"),
    ("dccuchile/bert-base-spanish-wwm-cased",       AutoModelForMaskedLM,               "Clases 2 y 4"),
    ("Qwen/Qwen3-0.6B",                             AutoModelForCausalLM,               "Clases 3 y 4"),
    ("mrm8488/bert-spanish-cased-finetuned-ner",    AutoModelForTokenClassification,    "Clase 4"),
    ("pysentimiento/robertuito-sentiment-analysis", AutoModelForSequenceClassification, "Clase 4"),
]

resultados = {}
for i, (nombre, clase, uso) in enumerate(MODELOS, 1):
    print(f"\n[{i}/{len(MODELOS)}] {nombre}  ({uso})")
    inicio = time.time()
    try:
        tok = AutoTokenizer.from_pretrained(nombre)
        modelo = clase.from_pretrained(nombre)
        n_param = sum(p.numel() for p in modelo.parameters())
        resultados[nombre] = {"uso": uso, "estado": "✅ listo", "parámetros": f"{n_param / 1e6:,.0f} M",
                              "segundos": round(time.time() - inicio)}
        print(f"   ✅ Listo · {n_param / 1e6:,.0f} M parámetros · {time.time() - inicio:.0f}s")
        del modelo, tok
        gc.collect()
    except Exception as e:
        resultados[nombre] = {"uso": uso, "estado": "❌ error", "parámetros": "—", "segundos": round(time.time() - inicio),
                              "error": f"{type(e).__name__}: {str(e)[:200]}"}
        print(f"   ❌ Error: {type(e).__name__}: {str(e)[:200]}")
        print("   → Revisa tu internet y vuelve a ejecutar esta celda.")

## 4. Prueba de fuego: ¿MiniGPT (Qwen3) responde?

In [ ]:
# Carga Qwen3 desde tu disco (ya no necesita internet) y le pide una respuesta corta.
NOMBRE_LLM = "Qwen/Qwen3-0.6B"
velocidad = None
if resultados.get(NOMBRE_LLM, {}).get("estado", "").startswith("✅"):
    tok = AutoTokenizer.from_pretrained(NOMBRE_LLM)
    llm = AutoModelForCausalLM.from_pretrained(NOMBRE_LLM, dtype=torch.float32)
    mensajes = [{"role": "user", "content": "Preséntate en una sola frase como MiniGPT, el asistente de la clase de IA."}]
    texto = tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    inicio = time.time()
    with torch.no_grad():
        salida = llm.generate(**entrada, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
    seg = time.time() - inicio
    nuevos = salida.shape[1] - entrada["input_ids"].shape[1]
    velocidad = nuevos / seg
    print("🤖", tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print(f"\n⏱️ Tu computador genera ≈ {velocidad:.1f} tokens por segundo.")
    print("✅ Rápido: la clase irá fluida." if velocidad >= 8 else "🐢 Algo lento: funcionará, pero las respuestas largas tardarán. Ten paciencia con el 'modo pensar'.")
    del llm, tok
    gc.collect()
else:
    print("⚠️ Qwen3 no se descargó. Vuelve a ejecutar la sección 3.")

## 5. Resumen final

In [ ]:
import pandas as pd

tabla = pd.DataFrame(resultados).T[["uso", "estado", "parámetros", "segundos"]]

# Tamaño real en tu disco (desde la caché de Hugging Face)
try:
    from huggingface_hub import scan_cache_dir
    tamanos = {r.repo_id: r.size_on_disk / 1e6 for r in scan_cache_dir().repos}
    tabla["en disco"] = [f"{tamanos[n]:,.0f} MB" if n in tamanos else "—" for n in tabla.index]
    total = sum(tamanos.get(n, 0) for n in tabla.index)
    print(f"Espacio usado por los modelos de la unidad: {total / 1000:.2f} GB")
except Exception:
    pass

display(tabla)
fallos = [n for n, r in resultados.items() if not r["estado"].startswith("✅")]
if not fallos:
    print("\n🎉 ¡Todo listo! Ya puedes llegar a clase. Los modelos funcionarán aunque no haya internet.")
else:
    print("\n⚠️ Faltan:", ", ".join(fallos))
    print("   Vuelve a ejecutar la sección 3. Si sigue fallando, mira la sección de problemas comunes abajo.")

## 🆘 Problemas comunes

| Mensaje / síntoma | Solución |
|---|---|
| `ModuleNotFoundError` después de instalar | *Kernel → Restart* y vuelve a ejecutar todo. |
| `ConnectionError`, `ReadTimeout`, se queda quieto | Internet inestable. Vuelve a ejecutar la sección 3: retoma lo que faltaba. |
| `does not recognize this architecture` / `qwen3` | Tu `transformers` es viejo: `pip install -U transformers` y reinicia el kernel. |
| Error sobre `torch.load` / versión de torch | Actualiza torch: `pip install -U torch` y reinicia el kernel. |
| `No space left on device` | Libera al menos 6 GB de disco. |
| El kernel se muere al cargar Qwen3 | Poca RAM: cierra programas. En clase usa `DTYPE = torch.bfloat16`. |
| Estoy en una red corporativa con proxy | Prueba desde otra red (por ejemplo, la de tu casa o datos del celular). |

**¿Dónde quedaron los modelos?** En la carpeta que mostró la primera celda (en Windows: `C:\Users\TU_USUARIO\.cache\huggingface`). No la borres hasta terminar la unidad.